# Test ingestion + retrieval endpoints

Start the service first:

```powershell
uv run python main.py
```

In [1]:
import json
import requests

BASE_URL = "http://localhost:9000"

def pretty(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False))

## Health

In [ ]:
r = requests.get(f"{BASE_URL}/health", timeout=10)
r.raise_for_status()
pretty(r.json())

## Ingest a batch of YouTube videos (up to 2 in flight server-side)

In [2]:
VIDEO_URLS = [
    "https://www.youtube.com/watch?v=r6EXZcTJyaA",
    "https://www.youtube.com/watch?v=gaa_5m3cmSc",
]

r = requests.post(f"{BASE_URL}/ingest", json={"urls": VIDEO_URLS}, timeout=120)
r.raise_for_status()
ingest_result = r.json()
pretty(ingest_result)

VIDEO_IDS = [item["video_id"] for item in ingest_result["results"] if item["video_id"]]
print("VIDEO_IDS =", VIDEO_IDS)

{
  "results": [
    {
      "url": "https://www.youtube.com/watch?v=r6EXZcTJyaA",
      "success": true,
      "video_id": "r6EXZcTJyaA",
      "title": "\"I don't like Programming\" | Prime Reacts",
      "channel": "ThePrimeagenHighlights",
      "chunks_indexed": 57,
      "already_indexed": false,
      "error": null
    },
    {
      "url": "https://www.youtube.com/watch?v=gaa_5m3cmSc",
      "success": true,
      "video_id": "gaa_5m3cmSc",
      "title": "The Irishman - Al Pacino Says You're Late Clip | Netflix",
      "channel": "Still Watching Netflix",
      "chunks_indexed": 8,
      "already_indexed": false,
      "error": null
    }
  ]
}
VIDEO_IDS = ['r6EXZcTJyaA', 'gaa_5m3cmSc']


## Retrieve: metadata mode

For queries like "which video performed better?" — returns per-video metadata only, no chunks.

In [3]:
VIDEO_IDS = ['r6EXZcTJyaA']
r = requests.post(
    f"{BASE_URL}/retrieve",
    json={"mode": "metadata", "video_ids": VIDEO_IDS},
    timeout=30,
)
r.raise_for_status()
pretty(r.json())

{
  "mode": "metadata",
  "query": null,
  "results": [],
  "metadata": [
    {
      "video_id": "r6EXZcTJyaA",
      "title": "\"I don't like Programming\" | Prime Reacts",
      "channel": "ThePrimeagenHighlights",
      "duration": 1594,
      "upload_date": "20260517",
      "view_count": 184273,
      "like_count": 3626
    }
  ]
}


## Retrieve: chunks mode (hybrid + RRF + cross-encoder rerank)

Pipeline: Dense (Qdrant) + BM25 → RRF → top-15 candidates → cross-encoder rerank → top-3.
Pass `video_ids` to scope retrieval to a subset of indexed videos.

In [6]:
QUERY = "timestamp where he says he will kidnap his granddaughter"
VIDEO_IDS = ['r6EXZcTJyaA', 'gaa_5m3cmSc']
r = requests.post(
    f"{BASE_URL}/retrieve",
    json={
        "mode": "chunks",
        "query": QUERY,
        "video_ids": VIDEO_IDS or None,
        "candidate_k": 15,
        "top_k": 3,
    },
    timeout=60,
)
r.raise_for_status()
retrieval = r.json()
pretty(retrieval)

{
  "mode": "chunks",
  "query": "timestamp where he says he will kidnap his granddaughter",
  "results": [
    {
      "rerank_score": -1.249232292175293,
      "rrf_score": 1.0,
      "text": "-[all agreeing] What do you want me to do? I said, \"you people.\" What do you want me to do,\napologize for it? That's exactly what I want,\nJimmy. An apology. I'll apologize for it. That's all I want. After you apologize for being late, you motherfucking wop cocksucker. [laughs] Jimmy, are you\nout of your fucking mind? I'll apologize for being late after I kidnap your granddaughter,\nrip her guts out, and send them to you\nin a fucking envelope! [all clamoring] [Tony Pro] Get him off! Come on! I'll fucking kill him! [Tony Jack] Come on, Tony! Jesus!",
      "metadata": {
        "start_time": 241.999,
        "end_time": 323.25,
        "video_id": "gaa_5m3cmSc",
        "title": "The Irishman - Al Pacino Says You're Late Clip | Netflix",
        "channel": "Still Watching Netflix",
        

In [ ]:
for i, hit in enumerate(retrieval["results"], start=1):
    md = hit["metadata"]
    snippet = hit["text"][:220].replace("\n", " ")
    print(
        f"#{i} rerank={hit['score']:.4f} rrf={hit['rrf_score']:.4f} "
        f"dense={hit['dense_score']} bm25={hit['bm25_score']}  "
        f"[{md.get('start_time')}-{md.get('end_time')}]  {md.get('title')}"
    )
    print(f"     {snippet}...\n")

## Sanity sweep across a few queries

In [ ]:
queries = [
    "what is the main topic of the video",
    "any mention of performance or optimization",
    "opening hook or introduction",
]

for q in queries:
    r = requests.post(
        f"{BASE_URL}/retrieve",
        json={"mode": "chunks", "query": q, "candidate_k": 15, "top_k": 3},
        timeout=60,
    )
    r.raise_for_status()
    res = r.json()["results"]
    print(f"\n=== {q} ===")
    for i, hit in enumerate(res, start=1):
        md = hit["metadata"]
        snippet = hit["text"][:160].replace("\n", " ")
        print(f"  #{i} rerank={hit['score']:.4f} [{md.get('start_time')}-{md.get('end_time')}] {snippet}...")